# PREVOĐENJE PROGRAMSKIH JEZIKA - 4. laboratorijska vježba

Svrha ove bilježnice je provesti vas kroz jednu od mogućnosti simulacije generiranih kodova i provjere njihove ispravnosti.

## Upute za postavljanje okruženja

Za ispravno izvođenje ove bilježnice potrebno je pratiti sljedeće korake:

1. **Preuzimanje alata**: Preuzmite arhivu sa stranice ARM GNU Toolchain. Korisnici Windows sustava trebaju preuzeti: arm-gnu-toolchain-15.2.rel1-mingw-w64-x86_64-arm-none-eabi.zip.

2. **Konfiguracija direktorija**: Raspakirajte arhivu i mapu bin postavite izravno u direktorij u kojem se nalazi ova bilježnica.

3. **Instalacija biblioteka**: Potrebno je instalirati Python biblioteku unicorn. To možete učiniti pokretanjem ćelije koja se nalazi neposredno ispod ovih uputa.

4. **Priprema testova**: Preuzmite arhivu s testovima sa službene stranice predmeta. Sve testne mape kopirajte u direktorij tests unutar vašeg radnog direktorija.

**Očekivana struktura direktorija**

Prije izvršavanja bilježnice, osigurajte da vaš radni direktorij izgleda ovako:

<pre>

radni_direktorij/
├── bin/                    # Mapa s ARM alatima
├── GeneratorKoda           # Izvorni kod vašeg generatora
├── tests/                  # Mapa s testnim primjerima
│   ├── 01_ret_broj/        # Pojedinačni testni slučaj
│   │   ├── test.in         # Ulazni podaci
│   │   └── test.out        # Očekivani izlaz
│   └── ...                 # Ostali testovi
└── simulator.ipynb         # Ova bilježnica

</pre>

In [1]:
%pip install unicorn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Simulacija

In [ ]:
import os
import subprocess
import tempfile
import glob

import unicorn.arm_const as arm
from unicorn import Uc, UC_ARCH_ARM, UC_MODE_ARM, UC_HOOK_INTR, UC_ERR_READ_UNMAPPED

ARM_CODE_ADDRESS = 0x10000
ARM_CODE_SIZE = 0x4000
MEMORY_SIZE = 0x10000

# Updated path handling
arm_toolchain_path = os.path.join(os.getcwd(), "bin")

if arm_toolchain_path not in os.environ["PATH"]:
    os.environ["PATH"] = arm_toolchain_path + os.pathsep + os.environ["PATH"]

def hook_intr(mu, intno, user_data):
    if intno == 2:
        mu.emu_stop()   

def evaluate_r6(arm_code_path, expected_r6_value):
    with tempfile.TemporaryDirectory() as tmpdir:
        asm_file = os.path.join(tmpdir, "code.s")
        obj_file = os.path.join(tmpdir, "code.o")
        elf_file = os.path.join(tmpdir, "code.elf")
        bin_file = os.path.join(tmpdir, "code.bin")

        try:
            with open(arm_code_path, "r") as src, open(asm_file, "w") as dst:
                dst.write(src.read())

            subprocess.run(
                ["arm-none-eabi-as", asm_file, "-o", obj_file], capture_output=True, check=True, text=True
            )

            subprocess.run([
                "arm-none-eabi-ld",
                "-Ttext", hex(ARM_CODE_ADDRESS),
                obj_file, "-o", elf_file
            ], capture_output=True, check=True, text=True)

            subprocess.run(["arm-none-eabi-objcopy", "-O", "binary", elf_file, bin_file], capture_output=True, check=True, text=True)
        except subprocess.CalledProcessError as e:
            return False, None, e.stderr
        except FileNotFoundError as e:
            return False, None, f"Tool not found: {e}"

        with open(bin_file, "rb") as f:
            code = f.read()

    mu = Uc(UC_ARCH_ARM, UC_MODE_ARM)
    mu.hook_add(UC_HOOK_INTR, hook_intr)

    mu.mem_map(0x0000, 0x30000)

    mu.mem_write(ARM_CODE_ADDRESS, code)

    mu.reg_write(arm.UC_ARM_REG_SP, 0x20000)

    try:
        # Timeout after 2 seconds (2000000 microseconds)
        mu.emu_start(ARM_CODE_ADDRESS, ARM_CODE_ADDRESS + len(code) + 0x1000, timeout=2000000)
    except Exception as e:
        print(f"Emulation error or timeout: {e}")
        return False, None, str(e)

    r6_value = mu.reg_read(arm.UC_ARM_REG_R6)

    if r6_value & 0x80000000:
        r6_signed = r6_value - 0x100000000
    else:
        r6_signed = r6_value

    print(f"  > R6 result: {hex(r6_value)} (Signed: {r6_signed}) (Expected: {expected_r6_value})")

    return r6_signed == expected_r6_value, r6_signed, None


TESTS_FOLDER = "tests"


def run_all_tests():
    if not os.path.exists(TESTS_FOLDER): 
        print(f"Tests folder {TESTS_FOLDER} not found.")
        return

    test_dirs = sorted([f for f in os.listdir(TESTS_FOLDER) if os.path.isdir(os.path.join(TESTS_FOLDER, f))])
    total = 169
    passed = 0
    for test_dir in test_dirs:
        test_path = os.path.join(TESTS_FOLDER, test_dir)
        print(f"Running {test_path}...")

        # Find .in and .out files
        in_files = glob.glob(os.path.join(test_path, "*.in"))
        out_files = glob.glob(os.path.join(test_path, "*.out"))
        
        if not in_files or not out_files:
            print(f"  Skipping {test_path}: .in or .out file missing")
            continue
            
        input_path = in_files[0]
        output_path = out_files[0]

        try:
            # java
            subprocess.run(f"javac GeneratorKoda.java", shell=True, check=True, capture_output=True, text=True)
            # Java prog now creates a.s directly
            result = subprocess.run(
                f"java -cp .. GK.GeneratorKoda < \"{input_path}\"",
                shell=True, check=True, stderr=subprocess.PIPE, text=True
            )
        except subprocess.CalledProcessError as e:
            print("Postoji greška u prevođenju koda:")
            if hasattr(e, 'stderr') and e.stderr: print(e.stderr)
            elif hasattr(e, 'output') and e.output: print(e.output)
            continue

        with open(output_path, "r") as f:
            line = f.read().strip()
            if not line: expected_val = 0
            else: expected_val = int(line.strip(), 10)

        success, actual_r6, error_msg = evaluate_r6("a.s", expected_val)
        
        if success:
            print("  [PASS]")
            passed += 1
        else:
            print("  [FAIL]")
            # Save artifacts
            if os.path.exists("a.s"):
                with open(os.path.join(".", "a.s"), "r") as f1:
                    content = f1.read()
                
                with open(os.path.join(".", f"fail_{test_dir}.s"), "w") as f2:
                    f2.write(content)
                    f2.write(f"\n\n@ Expected R6: {expected_val}\n")
                    if actual_r6 is not None:
                         f2.write(f"@ Actual R6:   {actual_r6}\n")
                    if error_msg:
                        f2.write(f"@ Error: {error_msg}\n")

    print("-" * 20)
    print(f"FINAL RESULT: {passed}/{total} tests passed.")


if __name__ == "__main__":
    run_all_tests()

Running tests\01_ret_broj...
  > R6 result: 0x1f (Signed: 31) (Expected: 31)
  [PASS]
Running tests\02_ret_global...
  > R6 result: 0x48 (Signed: 72) (Expected: 72)
  [PASS]
Running tests\03_veliki_broj...
  > R6 result: 0x499602d2 (Signed: 1234567890) (Expected: 1234567890)
  [PASS]
Running tests\04_neg_broj...
  > R6 result: 0xffffffac (Signed: -84) (Expected: -84)
  [PASS]
Running tests\05_plus...
  > R6 result: 0x1f (Signed: 31) (Expected: 31)
  [PASS]
Running tests\06_plus_signed...
  > R6 result: 0x3 (Signed: 3) (Expected: 3)
  [PASS]
Running tests\07_minus...
  > R6 result: 0x3 (Signed: 3) (Expected: 3)
  [PASS]
Running tests\08_bitor...
  > R6 result: 0x5b (Signed: 91) (Expected: 91)
  [PASS]
Running tests\09_bitand...
  > R6 result: 0x1 (Signed: 1) (Expected: 1)
  [PASS]
Running tests\10_bitxor...
  > R6 result: 0x5a (Signed: 90) (Expected: 90)
  [PASS]
Running tests\11_fun1...
  > R6 result: 0x7b (Signed: 123) (Expected: 123)
  [PASS]
Running tests\12_fun2...
  > R6 result: 0